# 讲课笔记 — Day 2 下午：LoRA、对齐与评测

---

**总时长：约 3.5 小时（含休息）**

**讲师用：逐 cell 教学指南，含时间分配、话术、常见问题**

---

## 整体节奏概览

| 时间段 | 内容 | 类型 | 节奏 |
|--------|------|------|------|
| 0:00-0:15 | Part 1: 为什么要 LoRA？显存数学 | 讲解 | 慢讲，算数学 |
| 0:15-0:45 | Part 2: LoRA 实现 + 练习1 | 代码+练习 | 边写边讲 |
| 0:45-1:15 | Part 2续: PEFT库 + 练习2 | 代码+练习 | 演示为主 |
| 1:15-1:25 | 休息 | | |
| 1:25-1:55 | Part 3: 量化+QLoRA + 练习3 | 代码+练习 | 重点在表格 |
| 1:55-2:30 | Part 4: DPO对齐 + 练习4 | 代码+练习 | 公式慢推 |
| 2:30-3:00 | Part 5: 模型评测 + 练习5 | 代码+练习 | 对比看结果 |
| 3:00-3:25 | Part 6: 企业决策 + 练习6 | 讲解+讨论 | 互动讨论 |
| 3:25-3:30 | Day2 总结 | 讲解 | 快速回顾 |

---

## 开场（cell-000）

**时间：2 分钟**

**对应 cell：** cell-000（课程路线图 markdown）

**操作：** 直接展示，不需要运行

### 话术

"好，大家下午好！上午我们学了 SFT 监督微调，学会了怎么让模型'做事'。下午我们要解决三个更实际的问题：

1. **成本问题**——全量微调太贵了，LoRA 怎么省钱？
2. **质量问题**——模型会做事但回答不好看，DPO 怎么'教品味'？
3. **评估问题**——怎么知道模型好不好？

最后我们会做一个企业决策分析，帮你回去跟老板汇报该怎么选方案。"

### 要点
- 快速过路线图表格，让学员知道今天的节奏
- 强调有 6 个练习，都是动手的
- 提醒中间有 10 分钟休息

---

## cell-001：环境初始化

**时间：1 分钟**

**操作：** 直接运行 cell-001，不需要讲解细节

### 话术

"先把环境跑起来，这个 cell 就是设置一些环境变量和 import，跟上午一样。大家确保没有报错就行。"

### 注意
- 如果有学员报 import 错误，检查是否装了 peft 库
- `HF_ENDPOINT` 设置了镜像，国内环境必须的

---

# Part 1：为什么要 LoRA？（15 分钟）

## cell-002 ~ cell-005：GPU 显存数学

**时间：15 分钟（这是全课最重要的铺垫，要讲透）**

**对应 cell：** cell-002（显存分析 markdown）、cell-003（import+设备检测）、cell-004（显存计算代码）、cell-005（可视化图表）

**操作：** 先讲 cell-002 的 markdown，然后运行 cell-003/004/005

### 核心概念：全量微调的显存灾难

**在白板或 PPT 上慢慢算这笔账：**

```
7B 模型全量微调 FP16 需要多少显存？

① 模型参数：7B × 2 bytes = 14 GB
② 梯度：     7B × 2 bytes = 14 GB（每个参数都要存梯度）
③ Adam 优化器：7B × 8 bytes = 56 GB（m + v 各 4 bytes）
④ 激活值：    约 7 GB（取决于 batch size）

总计：14 + 14 + 56 + 7 ≈ 91 GB！

RTX 3090 = 24 GB → 需要 3.8 张！
A100 80GB → 勉强 1 张，但 batch size 很小
```

### 话术

"大家看，为什么全量微调贵？不是模型本身大，**是优化器吃的显存最多**。Adam 优化器要给每个参数存两份状态——动量 m 和方差 v，每份 4 bytes。7B 参数 × 8 bytes = 56GB，光优化器就比模型本身大 4 倍！

这就是为什么我们需要 LoRA——不是所有参数都需要训练的。如果我们只训练 1% 的参数，优化器开销直接缩小 100 倍。"

### 运行 cell-004 后指出
- 看输出的表格，`Full FP16` 的 91GB vs `LoRA FP16` 的 21.7GB
- 重点：优化器从 56GB 降到 0.6GB，**这是 LoRA 省钱的核心**

### 运行 cell-005 后指出
- 柱状图中，蓝色（Optimizer）是最大的一块
- LoRA 和 QLoRA 的蓝色几乎看不到了

### 常见学员问题

**Q: 为什么 Adam 要存两份状态？SGD 不行吗？**
A: SGD 只需要 4 bytes/参数（只存动量），但收敛慢很多。Adam 效果好是因为它自适应学习率，但代价就是 2x 的状态。实际中 AdamW 8-bit 可以省一半。

**Q: 激活值为什么也占这么多？**
A: 反向传播需要保存前向传播的中间结果。可以用 gradient checkpointing 来省激活显存，但会慢 20-30%。

**Q: 我公司只有 RTX 4090 (24GB)，能微调什么模型？**
A: QLoRA INT4 的 7B 模型，只需要 6GB！后面我们会讲到。

### 过渡

"好，既然全量微调这么贵，LoRA 是怎么做到只训练 1% 参数的？下面我们从数学公式开始，然后手写一个 LoRA 层。"

---

# Part 2：LoRA 实现（30 分钟）

## cell-006：LoRA 核心公式（Markdown）

**时间：5 分钟**

**操作：** 展示 markdown，边读边在白板画图

### 核心概念

LoRA 的核心公式：
$$y = Wx + BAx \cdot \frac{\alpha}{r}$$

### 话术

"LoRA 的想法特别简单——微调的时候，权重变化 ΔW 其实是'低秩'的。什么意思？就是说，虽然权重矩阵很大（比如 4096×4096），但微调带来的变化其实只涉及少数几个'方向'。

打个比方：你有一本百科全书（原始权重 W），现在要教模型做客服。你不需要重写整本书，只需要在旁边贴个便签纸（ΔW = BA）。这个便签纸可以分解成两个小矩阵的乘积：

- **A 矩阵**（rank × in_features）：把高维输入'压缩'到低维
- **B 矩阵**（out_features × rank）：再'解压'回高维

rank=8 的时候，4096×4096 = 1600万参数，变成 A(8×4096) + B(4096×8) = 65536 参数，**省了 99.6%！**"

### 要画的图

```
输入 x ──→ [W(冻结)] ──→ Wx ──→ (+) ──→ 输出 y
      │                          ↑
      └──→ [A(降维)] ──→ [B(升维)] ──→ × α/r
           (rank×in)    (out×rank)
```

### 指出表格
- rank=8 时节省 99.6%
- rank 越大表达力越强，但参数越多
- 实践中 rank=8~64 覆盖绝大部分场景

### 关键：初始化策略
- A 用 Kaiming 初始化（有值）
- **B 初始化为零** → 训练开始时 ΔW = B×A = 0
- "为什么 B 要初始化为零？因为我们不想一上来就破坏原模型的行为！训练刚开始时 LoRA 的贡献是零，然后慢慢学。"

---

## cell-007：手写 LoRALinear 类

**时间：8 分钟**

**对应 cell：** cell-007（LoRALinear 类定义 + 测试）

**操作：** 运行 cell-007，逐段讲解代码

### 话术

"好，看代码。这个 LoRALinear 类就是我们刚才说的公式的实现。我来带大家过一遍关键部分：

首先看 `__init__`：
1. 创建一个普通的 `nn.Linear`，然后把它的 `requires_grad` 设为 `False`——**冻结原始权重**
2. 创建 `lora_A`，shape 是 `(rank, in_features)`——降维矩阵
3. 创建 `lora_B`，shape 是 `(out_features, rank)`——升维矩阵，**注意初始化为零**

然后看 `forward`：
1. `original_output = self.linear(x)` — 原始路径，冻结的
2. `F.linear(x, self.lora_A)` — 降维：输入 x 从 in_features 变成 rank
3. `F.linear(result, self.lora_B)` — 升维：从 rank 变回 out_features
4. 最后乘 `scaling = alpha/rank`，加到原始输出上"

### 运行后指出输出

```
Trainable params: 8,192      ← 只有这么点需要训练！
Frozen params:    262,656    ← 原始权重全部冻结
Ratio: 3.12%                ← 只训练 3% 的参数
```

"大家看，512×512 的矩阵有 26 万参数，LoRA 只增加了 8192 个可训练参数。训练的时候梯度、优化器状态都只给这 8192 个参数算，省了 97% 的训练开销！"

### 补充：merge_weights
"还有一个重要的方法 `merge_weights`——推理的时候，把 LoRA 的权重合并回原始权重，推理速度跟原模型一模一样，零额外开销。这是 LoRA 的另一个优点。"

### 常见问题

**Q: F.linear(x, A) 和 x @ A.T 有什么区别？**
A: 完全等价。`F.linear(x, A)` 就是 `x @ A.T`，PyTorch 的 Linear 层就是用这个实现的。

**Q: alpha 和 rank 的关系是什么？**
A: `scaling = alpha / rank`。通常 alpha 设成 rank 的 2 倍（比如 rank=8, alpha=16），这样 scaling=2。alpha 越大，LoRA 的影响越大。

---

## 练习 1：实现 lora_forward()（10 分钟）

**对应 cell：** cell-008（题目 markdown）、cell-009（代码框）、cell-010（验证）

**时间：10 分钟（5分钟做 + 5分钟讲解）**

### 话术（发练习前）

"好，现在轮到你们了。练习 1 是实现一个 `lora_forward` 函数。跟刚才那个 LoRALinear 类的 forward 方法一样的逻辑，就是拆成独立函数。

三步：
1. `base = W(x)` — 原始线性层的输出
2. 先 `F.linear(x, lora_A)` 降维，再 `F.linear(结果, lora_B)` 升维
3. `return base + lora_out * scaling`

大家动手，5 分钟时间。做完运行下面的验证 cell，看到 'All tests passed!' 就对了。"

### 巡场要点
- 常见错误 1：`F.linear(x, lora_B)` 写反了，应该先 A 后 B
- 常见错误 2：忘了乘 scaling
- 常见错误 3：把 `W(x)` 写成 `F.linear(x, W)` — W 是 nn.Linear 对象，直接调用即可

### 讲解答案

"答案其实就三行：
```python
base = W(x)
lora_out = F.linear(F.linear(x, lora_A), lora_B)
return base + lora_out * scaling
```
验证测试检查了两个边界条件：B=0 时增量为 0，scaling=0 时增量也为 0。这确保了初始化行为正确。"

### 过渡
"手写 LoRA 帮助理解原理，但实际项目中谁会手写？下面我们用 HuggingFace 的 PEFT 库，3 行代码搞定。"

---

## cell-011 过渡 + cell-012：PEFT 库介绍

**时间：2 分钟**

**对应 cell：** cell-011（过渡 markdown）、cell-012（PEFT 介绍 markdown）

### 话术

"实际项目中我们用 PEFT 库（Parameter-Efficient Fine-Tuning）。只需要 3 步：
1. `LoraConfig(...)` — 写配置
2. `get_peft_model(model, config)` — 一行注入 LoRA
3. 正常训练

就这么简单。"

---

## cell-013：加载中文 GPT-2 模型

**时间：2 分钟（含等待加载）**

**对应 cell：** cell-013

**操作：** 运行，等模型加载完

### 话术

"我们用一个中文 GPT-2 模型来做演示。这个模型 1.18 亿参数，很小，但够用来演示 LoRA 的效果。"

### 运行后指出
- `参数量: 118,295,040` — 1.18 亿，比 7B 小很多，但原理一样
- 如果有 warning 可以忽略（代码里已经 suppress 了大部分）

---

## cell-014：LoRA 配置 + 注入

**时间：5 分钟（重点讲配置参数）**

**对应 cell：** cell-014

**操作：** 运行，讲解每个配置参数

### 话术

"来看 LoRA 配置，每个参数什么意思：

- `r=16` — rank，秩。越大表达力越强，但参数也越多。一般 8~64
- `lora_alpha=32` — 缩放因子，通常设成 rank 的 2 倍
- `lora_dropout=0.1` — 防止过拟合，数据少的时候可以设大点
- `target_modules=["c_attn", "c_proj"]` — **关键！** 这决定了 LoRA 加在哪些层上。GPT-2 的注意力层叫 c_attn 和 c_proj。不同模型名字不同，LLaMA 的叫 q_proj, v_proj 等
- `bias="none"` — 不训练 bias，省参数"

### 运行后指出

```
trainable params: 1,622,016 || all params: 119,917,056 || trainable%: 1.3526
```

"看！1.19 亿参数的模型，LoRA 只训练 162 万参数，占 1.35%。训练速度快、显存省、效果还不错。"

### 常见问题

**Q: target_modules 怎么知道该选哪些？**
A: 一般选注意力层的 Q、K、V 投影。可以用 `model.named_modules()` 打印出来看。PEFT 也支持 `target_modules="all-linear"` 一键全选。

**Q: rank 应该设多大？**
A: 经验法则：简单任务（分类、情感分析）rank=8 就够；复杂任务（代码生成、长文写作）可以用 32~64。后面的练习 2 我们会对比不同 rank 的效果。

---

## cell-015 ~ cell-016：数据准备 + 训练

**时间：5 分钟**

**对应 cell：** cell-015（数据集构建）、cell-016（LoRA 训练循环）

**操作：** 快速运行，不需要逐行讲数据构造代码

### 话术

"这里我们用一个客服分类的模拟数据集——1200 条样本，6 个类别：延迟发货、退款申请、地址修改等等。

数据格式很简单：`文本：xxx\n类别：` 作为 prompt，标签作为 answer。训练的时候只在标签部分算 loss（prompt 部分的 labels 设成 -100）。

这个训练只跑 2 个 epoch，主要是演示流程。"

### 运行 cell-016 后指出
- 看 loss 下降趋势
- 训练速度 vs 上午的全量微调速度对比
- 如果有 GPU，几十秒就跑完；CPU 可能要几分钟

### 小技巧
"实际项目中不会这样手写训练循环，会用 Trainer。但手写的好处是你能看到每一步在做什么。"

---

## 练习 2：对比 rank=4/16/32 的效果（10 分钟）

**对应 cell：** cell-017（题目 markdown）、cell-018（代码框）、cell-019（可视化）

**时间：10 分钟（5分钟做 + 5分钟讲解）**

### 话术（发练习前）

"练习 2 是对比不同 rank 的拟合能力。我们用一个简单的正弦函数拟合任务——不是 NLP，是故意用简单任务，这样你能直观看到 rank 的影响。

rank=4 能拟合多好？rank=16 呢？rank=32 呢？大家跑一下看看 MSE（均方误差）。"

### 运行后指出
- rank 越大，MSE 越小（拟合越好）
- 但 rank=4 到 rank=16 的提升很大，rank=16 到 rank=32 的提升变小了
- **边际递减效应**：rank 不是越大越好，到一定程度就饱和了
- 看可视化图：不同 rank 的拟合曲线

### 关键结论

"所以实际项目中，rank=8~16 是最常用的。rank 太大不仅参数多，还容易过拟合。"

### 过渡

"LoRA 减少了训练参数。但模型本身还是那么大——7B 模型 FP16 就要 14GB。能不能把模型本身也压缩？这就是量化。"

---

# 休息提醒（如果按时间表，这里应该休息 10 分钟）

"大家休息 10 分钟。回来我们讲量化和 QLoRA，然后是 DPO 对齐。"

---

---

# Part 3：量化 + QLoRA（20 分钟）

## cell-020：量化介绍（Markdown）

**时间：5 分钟**

**对应 cell：** cell-020（量化 markdown）

**操作：** 展示 markdown 表格，重点讲

### 核心概念

| 精度 | 字节/参数 | 7B 模型大小 | 相比 FP32 |
|------|----------|------------|----------|
| FP32 | 4 | 28 GB | 1x |
| FP16 | 2 | 14 GB | 2x 压缩 |
| INT8 | 1 | 7 GB | 4x 压缩 |
| INT4 | 0.5 | 3.5 GB | 8x 压缩 |

### 话术

"量化就像降低照片分辨率。原来每个参数用 32 bit 浮点数存，现在只用 8 bit 甚至 4 bit 整数存。

具体怎么做？对称量化很简单：
1. 找到这组参数的最大绝对值 max_val
2. 计算 scale = max_val / 127（INT8 的范围是 -128 到 127）
3. 每个参数 ÷ scale 再四舍五入，变成整数
4. 用的时候 × scale 还原回浮点数

精度会有损失，但对于大模型来说，这个损失几乎不影响效果。"

### 要强调
- **QLoRA = INT4 量化 + LoRA**
- 模型权重用 4-bit 存储（省空间），LoRA 参数用 FP16 训练（保精度）
- 这让 24GB 的 RTX 4090 也能微调 7B 模型！

---

## cell-021：手写对称量化

**时间：5 分钟**

**对应 cell：** cell-021（量化代码 + 对比）

**操作：** 运行，看输出

### 话术

"我们来看实际量化的效果。这段代码对同一个权重矩阵做了 8-bit 和 4-bit 量化："

### 运行后指出

```
8-bit 量化：4x 压缩，误差 0.009 — 几乎无损！
4-bit 量化：8x 压缩，误差 0.165 — 有损但可接受
```

"看这个对比：
- 8-bit 的误差只有 0.009，基本上看不出区别
- 4-bit 的误差大了一些（0.165），但在大模型里这个程度的误差对最终效果影响很小

为什么 4-bit 还能 work？因为大模型有冗余——很多参数的精确值并不重要，大致方向对就行。"

### 常见问题

**Q: 量化后能还原回去吗？**
A: 可以近似还原（dequantize），但精度已经丢了。就像 JPEG 压缩，不可逆。

**Q: 量化只用于推理还是训练也行？**
A: 模型权重量化后用于前向传播，但 LoRA 的参数和梯度仍然用 FP16。所以训练精度不受影响。

---

## 练习 3：GPU 显存估算器（10 分钟）

**对应 cell：** cell-022（题目 markdown）、cell-023（代码框 + 验证）

**时间：10 分钟（5分钟做 + 5分钟讲解）**

### 话术（发练习前）

"练习 3 很实用——写一个显存估算器。输入模型大小、精度、可训练比例，输出需要多少 GB 显存。

公式：
- model_gb = 参数量(B) × 每参数字节数
- gradient_gb = 可训练参数量(B) × 2（FP16 梯度）
- optimizer_gb = 可训练参数量(B) × 8（Adam 的 m+v）
- activation_gb = model_gb × 0.5（经验值）

大家试试。"

### 运行后指出——这是全课最震撼的输出！

```
Scenario                Model     Grad    Optim      Act    Total
------------------------------------------------------------------------
7B Full FP16            14.0G    14.0G    56.0G     7.0G    91.0G
7B LoRA FP16            14.0G     0.1G     0.6G     7.0G    21.7G
7B QLoRA INT4            3.5G     0.1G     0.6G     1.8G     6.0G
13B QLoRA INT4           6.5G     0.1G     0.5G     3.2G    10.4G
```

### 话术（讲解时，要激动！）

"大家看这个表！

- **全量 FP16：91GB**——需要 2 张 A100 80GB！
- **LoRA FP16：21.7GB**——一张 RTX 3090 就够了！省了 4 倍！
- **QLoRA INT4：6GB！**——一张 RTX 4060 都能跑！从 91GB 降到 6GB，省了 15 倍！

这就是为什么 QLoRA 是中小企业的福音——不需要买几十万的 A100 服务器，一张消费级显卡就能微调 7B 模型。

甚至 13B 的 QLoRA 也只要 10.4GB，一张 RTX 3060 12GB 就能搞定！"

### 常见问题

**Q: 这个估算准吗？**
A: 大方向对，实际会因 batch size、序列长度、是否用 gradient checkpointing 等因素有偏差。但数量级是对的。

### 过渡

"好，到这里我们解决了'成本问题'——LoRA+量化让微调变得便宜。但微调只解决了'能力'问题。接下来我们要解决'品味'问题——让模型学会什么样的回答是好的。这就是 DPO 对齐。"

---

# Part 4：DPO 对齐（30 分钟）

## cell-027：为什么需要对齐 + RLHF vs DPO

**时间：5 分钟**

**对应 cell：** cell-027（markdown）

**操作：** 展示 markdown，重点讲对比表

### 话术

"SFT 教模型'怎么做题'，对齐教模型'怎么做得漂亮'。

打个比方：SFT 就像教一个新员工业务流程，DPO 就像给他看标准答案和反面教材的对比——'这样回答客户是好的，那样回答是不好的'。

以前做对齐要用 RLHF——先训练一个奖励模型，再用 PPO 强化学习。这个流程巨复杂，超参数特别多，训练不稳定，容易崩。

DPO 说：我不需要奖励模型！我直接从偏好数据里学！"

### 重点指出对比表

| 维度 | RLHF | DPO |
|------|------|-----|
| 需要几个模型 | 3个 | 2个 |
| 实现复杂度 | 很高 | 低（类似 SFT） |
| 训练稳定性 | 不稳定 | 稳定 |

"**除非你是 OpenAI 级别的团队，否则 DPO 是更实际的选择。**"

---

## cell-028：DPO 核心公式

**时间：5 分钟（公式要慢推）**

**对应 cell：** cell-028（DPO 公式 markdown）

### 话术（要慢！）

"DPO 的公式看着复杂，但拆开看其实很直觉。我一步步来：

**输入是什么？** 三元组：(prompt, chosen回答, rejected回答)

**Loss 怎么算？**

第一步：算 log-ratio
- chosen 的 log-ratio = log(策略模型给chosen的概率) - log(参考模型给chosen的概率)
- rejected 的 log-ratio = 同理

第二步：算差值
- logits = chosen的log-ratio - rejected的log-ratio
- 这个值越大越好（说明模型更偏好 chosen）

第三步：算 loss
- loss = -logsigmoid(β × logits)
- β 控制偏离参考模型的程度，通常 0.1

**为什么需要参考模型？** 防止模型'跑偏'太远。如果没有参考模型的约束，模型可能为了追求 chosen 的高概率而把其他所有输出的概率都压到零。"

### 要在白板画的

```
策略模型给 chosen 打分高 ──┐
                          ├──→ 差值大 ──→ loss 小 ✓
策略模型给 rejected 打分低 ──┘

β 太大 → 偏离参考模型太多 → 模型退化
β 太小 → 学不到偏好 → 效果差
```

---

## cell-029：偏好数据

**时间：3 分钟**

**对应 cell：** cell-029（PREFERENCE_DATA 定义）

**操作：** 运行，看数据格式

### 话术

"看偏好数据长什么样：

- Prompt: '中国的首都是哪里？'
- Chosen: '北京。' ← 简洁、直接
- Rejected: '中国的首都是北京。北京是中华人民共和国的首都...' ← 啰嗦、废话多

注意：rejected 不是'错误答案'，而是'不够好的答案'。DPO 的本质是教模型风格偏好，不是对错。

在这个例子里，我们的偏好是'简洁'。实际企业场景里，偏好可以是：
- 更安全（不输出有害内容）
- 更友好（客服语气）
- 更专业（技术文档风格）"

### 要指出
- 只有 10 条数据，演示用
- 实际项目中需要至少几百条偏好数据
- 数据质量比数量重要

---

## cell-030：DPO Loss 实现

**时间：3 分钟**

**对应 cell：** cell-030（dpo_loss 函数）

**操作：** 运行，对照公式讲代码

### 话术

"看代码，跟公式一一对应：

```python
chosen_log_ratio = policy_chosen - reference_chosen      # chosen 的 log(π/π_ref)
rejected_log_ratio = policy_rejected - reference_rejected # rejected 的 log(π/π_ref)
log_ratio_diff = chosen_log_ratio - rejected_log_ratio   # 差值
losses = F.logsigmoid(beta * log_ratio_diff)             # sigmoid + log
loss = -losses.mean()                                    # 取负号
```

就这 5 行核心代码。"

### 运行后指出
- `loss=0.6444` — 初始 loss 接近 ln(2)=0.693，因为模型还没学会偏好
- `acc=100%` — 这是因为测试数据中 chosen 概率本来就高
- `margin=1.0` — chosen 和 rejected 的奖励差距

---

## cell-031 ~ cell-033：DPO 模型加载 + 辅助函数 + 训练

**时间：5 分钟**

**对应 cell：** cell-031（加载 Qwen2.5-0.5B）、cell-032（辅助函数）、cell-033（训练循环）

**操作：** 依次运行

### cell-031 运行后指出

"注意几个关键设计：
1. 加载了**两个一样的模型**：policy_model（会被训练）和 reference_model（冻结不动）
2. reference_model 所有参数 requires_grad=False
3. policy_model 也不是全量训练——只解冻了最后 4 层 + lm_head，只有 39.6% 的参数可训练

这就是'部分微调'——对齐不需要改太多参数。"

### cell-032 快速过
"辅助函数不用细看，就是 tokenize、生成回复、计算 log 概率这些工具函数。"

### cell-033 训练后指出
- 只训练了 1 个 epoch，10 条数据
- 看 loss 变化：从约 0.69 开始下降
- preference accuracy 提升
- "实际项目中会训练更多数据和 epoch"

### 常见问题

**Q: 为什么要两个模型？占 2 倍显存？**
A: 参考模型是约束，防止策略模型'跑偏'。如果显存紧张，可以用 LoRA——参考模型就是 base model，策略模型是 base + LoRA adapter，只多一份 LoRA 参数的显存。

**Q: DPO 数据怎么收集？**
A: 最常见的方式是让模型生成多个回答，人工标注哪个好。也可以用 GPT-4 当 judge 来自动标注。

---

## 练习 4：对比 DPO 训练前后（10 分钟）

**对应 cell：** cell-034（题目 markdown）、cell-035（代码框+验证）

**时间：10 分钟（5分钟做 + 5分钟讲解）**

### 话术（发练习前）

"练习 4 分两部分：
1. 对 5 个问题，分别用 reference_model 和 policy_model 生成回答，看有什么不同
2. 写一个 `eval_pref_acc` 函数，计算模型对 chosen 的偏好准确率

第一部分就是调用 `generate_response`，打印对比。
第二部分的思路：对每条数据，分别算 chosen 和 rejected 的 log 概率，如果 chosen > rejected 就算 correct。"

### 运行后指出

```
Preference Accuracy: Ref=10%, DPO=20%
```

"准确率提升不大？因为我们只用了 10 条数据训练了 1 个 epoch！这只是演示。实际项目中用几百条数据训练几个 epoch，准确率提升会很明显。

更重要的是看生成对比——DPO 训练后的模型回答是否更简洁、更直接了。"

### 过渡

"好，我们现在有了微调（SFT/LoRA）和对齐（DPO）。但怎么知道模型到底好不好？这就需要评测。"

---

# Part 5：模型评测（25 分钟）

## cell-036：评测体系介绍

**时间：5 分钟**

**对应 cell：** cell-036（评测体系 markdown）

**操作：** 展示 markdown

### 话术

"评测分三个层次：

**第一层：困惑度（PPL）**
- 衡量模型对文本的'困惑程度'
- PPL 越低越好
- 好的模型：PPL ≈ 10-20
- 垃圾模型：PPL > 1000
- 局限：只衡量'语言流畅度'，不衡量'回答是否正确'

**第二层：任务指标**
- BLEU：翻译质量（n-gram 重叠度）
- F1：信息抽取的准确率和召回率
- Exact Match：精确匹配（数学题等）
- 局限：需要标准答案

**第三层：LLM-as-Judge**
- 用 GPT-4 等强模型当'裁判'
- 多维打分：有用性、准确性、流畅性
- 最接近人类评判
- 局限：成本高、有偏见"

### 要强调
"三层不是替代关系，是互补的。实际项目中三个都要用。"

---

## cell-037 ~ cell-038：PPL 计算

**时间：5 分钟**

**对应 cell：** cell-037（PPL 代码）、cell-038（PPL 对比）

**操作：** 运行，看对比结果

### 话术

"PPL 的公式很简单：
$$PPL = \exp\left(-\frac{1}{N}\sum\log P(w_i)\right)$$

就是每个 token 概率的几何平均的倒数。模型越确定下一个 token 是什么，PPL 越低。"

### 运行后指出

```
好的文本 PPL ≈ 12    ← 模型读得懂，不困惑
坏的文本 PPL ≈ 1857  ← 模型一脸懵，完全看不懂
```

"这个对比非常直观。通顺的中文句子，模型 PPL 十几；乱码文本，PPL 上千。

但 PPL 有局限——'今天天气真好'和'今天天气真差'的 PPL 可能差不多，因为都是通顺的句子。所以 PPL 只衡量语言质量，不衡量内容质量。"

---

## cell-039：任务指标（BLEU、F1、Exact Match）

**时间：5 分钟**

**对应 cell：** cell-039（任务评测代码）

**操作：** 运行，快速过

### 话术

"BLEU 是翻译领域最常用的指标，计算 n-gram 重叠度。F1 是信息抽取常用的。Exact Match 就是答案完全一样。

这些是'有标准答案'的评测。但很多时候——比如开放式问答、创意写作——没有标准答案。这时候就需要 LLM-as-Judge。"

### 快速过就行
- BLEU 的 4-gram、brevity penalty 不用深讲
- 重点是让学员知道有这些指标

---

## cell-040：LLM-as-Judge

**时间：5 分钟**

**对应 cell：** cell-040（LLM Judge 代码）

**操作：** 运行，看 prompt 格式和结果

### 话术

"LLM-as-Judge 的思路很简单：把两个模型的回答给 GPT-4 看，让它打分。

看这个 prompt 模板——给 judge 看问题、Response A、Response B，让它从 Helpfulness、Accuracy、Clarity 三个维度评分，然后选 A 或 B 或 Tie。

这个方法现在越来越流行，Chatbot Arena 就是这么做的。"

### 运行后指出
- 这里用模拟数据，实际中要调 GPT-4 API
- 结果是概率分布：A 20%、B 70%、Tie 10%

### 常见问题

**Q: LLM-as-Judge 可靠吗？**
A: 有已知偏见——比如 GPT-4 倾向于选更长的回答、位置偏差（先看到的容易被选）。可以通过交换 A/B 顺序来缓解。

**Q: 能不能用自己训练的模型当 Judge？**
A: 可以，但 Judge 要比被评测的模型强。用弱模型评强模型不靠谱。

---

## 练习 5：评测框架（10 分钟）

**对应 cell：** cell-041（题目 markdown）、cell-042（代码框+验证）

**时间：10 分钟

### 话术

"练习 5 是用我们刚学的评测方法来评估模型。大家按照提示完成代码，运行验证。"

### 要点
- 这个练习相对简单，主要是调用前面定义的函数
- 帮助学员把评测方法串起来

### 过渡

"好，到现在我们学了微调（SFT/LoRA/QLoRA）、对齐（DPO）、评测（PPL/BLEU/LLM-Judge）。最后一个问题：**回到企业场景，到底该怎么选？** 这就是我们最后一部分——企业决策。"

---

# Part 6：企业决策（20 分钟）

## cell-045：决策树 + 成本表

**时间：10 分钟（最实用的部分，要讲透）**

**对应 cell：** cell-045（决策树 markdown）

**操作：** 展示 markdown 决策树

### 核心：按数据量决策

```
你有多少标注数据？
├── < 100 条 → 直接调 API（GPT-4/Claude），Few-shot 足够
├── 100-1K 条 → LoRA 微调小模型（7B），够学到基本模式
├── 1K-10K 条 → 全面 LoRA 微调，可以考虑 13B 模型
└── > 10K 条 → 考虑全量微调
      └── 有偏好数据？
            ├── 是 → 加 DPO
            └── 否 → 纯 SFT
```

### 话术

"这个决策树是我们今天所有内容的总结。大家回去跟老板汇报，可以直接用这个框架。

**第一个问题：有多少数据？**
- 100 条以下？别微调了，直接调 API，省事省钱
- 100-1000 条？LoRA 微调 7B 模型，用 QLoRA 在一张 4090 上就能跑
- 1000 以上？可以上更大模型，数据量够支撑

**第二个问题：需要对齐吗？**
- 如果用户反馈模型回答'不好看'（太啰嗦、不够专业等），就需要收集偏好数据做 DPO
- 不是所有场景都需要对齐。分类任务、信息抽取这类结构化任务，SFT 就够了"

### 成本表

| 方案 | 一次性成本 | 月度成本 | 适合场景 |
|------|----------|---------|----------|
| API 调用 | 0 | $0.01-0.06/1K tokens | 低量、探索期 |
| QLoRA 7B | ~$50 | ~$200/月 | 中等量，需定制 |
| Full FT 7B | ~$500 | ~$200/月 | 大量数据 |
| 部署 70B | ~$2000 | ~$2000/月 | 高质量要求 |

"注意 QLoRA 7B 的训练成本只要 $50！A100 租 4 小时就够。但推理是持续的——$200/月是一张 GPU 的云服务器费用。"

---

## cell-046：盈亏平衡计算器

**时间：5 分钟**

**对应 cell：** cell-046（break-even 计算代码）

**操作：** 运行，看三个场景对比

### 话术

"API vs 自部署，到底哪个划算？看这三个场景："

### 运行后指出

```
小型客服Bot(100次/天)    → API $45/月 vs 自部署 $200/月 → 用 API！
中型内容审核(1000次/天)  → API $900/月 vs 自部署 $400/月 → 自部署，0.4 个月回本！
大型数据处理(10000次/天) → API $7200/月 vs 自部署 $800/月 → 自部署，0.1 个月回本！
```

"规律很明显：
- **日请求量 < 几百次**：API 更便宜，因为自部署有固定的服务器成本
- **日请求量 > 1000 次**：自部署更便宜，而且回本特别快

这就是为什么大公司都在自己微调部署——请求量大的时候，API 费用是天文数字。"

### 常见问题

**Q: 自部署的隐性成本呢？人力、运维？**
A: 好问题！自部署确实有额外成本：需要 ML 工程师维护（人力）、需要处理模型更新、需要监控服务质量。小团队可能没这个能力。所以日请求量不到千的，建议先用 API。

**Q: 能不能先用 API 验证需求，再切换到自部署？**
A: 这是最佳实践！先用 API 跑 MVP（最小可行产品），验证需求后再微调自部署。

---

## 练习 6：电商客服 AI 决策分析（10 分钟）

**对应 cell：** cell-047（题目 markdown）、cell-048（代码框+验证）

**时间：10 分钟（5分钟做 + 5分钟讨论）**

### 话术（发练习前）

"最后一个练习是综合应用。场景是一个电商公司要部署中文客服 AI：
- 日均 5000 次请求
- 平均每次 600 tokens
- 有 3000 条标注对话 + 500 条偏好数据
- 预算首月 $2000

你要做四个决策：
1. 选什么模型？（7B？13B？70B？）
2. 用什么微调策略？（SFT？LoRA？QLoRA？要不要 DPO？）
3. 估算训练成本和月推理成本
4. 跟 API 方案对比，算盈亏平衡

大家动手算，然后我们一起讨论。"

### 参考答案讲解

"来看参考答案的思路：

**1. 模型选择：Qwen2.5-7B**
- 为什么不选 70B？预算不够，70B 训练要 $2000，月推理也要 $2000
- 为什么不选 1.8B？客服场景需要一定的理解和生成能力，太小不够
- 7B 是性价比最高的选择

**2. 微调策略：QLoRA SFT + DPO**
- 有 3000 条标注数据 → 够做 SFT
- 有 500 条偏好数据 → 加 DPO 让回答更符合公司风格
- 用 QLoRA 省显存，一张 4090 就能搞定

**3. 成本估算**
- 训练：$500（A100 一天）
- 月推理：$200（一张 A10G 云服务器）

**4. 盈亏平衡**"

### 运行后指出

```
月 API 成本: $2250
月自部署成本: $200
盈亏平衡: 0.24 个月！
每月节省: $2050
```

"0.24 个月就是大约 1 周回本！之后每个月省 $2050。一年省 $24600。这就是自部署的价值。

所以给老板的汇报很简单：首月投入 $700（$500训练+$200推理），1 周回本，之后每月省 $2050。"

---

# Day 2 总结（5 分钟）

## cell-050：总结

**对应 cell：** cell-050（总结 markdown）

**操作：** 展示知识图谱

### 话术

"来回顾一下今天学了什么：

```
上午 SFT → 下午 LoRA → DPO → 评测
全量参数    0.1%参数    偏好对齐   PPL/LLM-Judge

模型能做事 → 低成本微调 → 输出更对味 → 知道好不好
```

**三个关键公式：**
1. LoRA: $y = Wx + BAx \cdot \frac{\alpha}{r}$
2. DPO: $-\log\sigma(\beta(\Delta_{chosen} - \Delta_{rejected}))$
3. PPL: $\exp(-\frac{1}{N}\sum\log P(w_i))$

**企业决策金句：**
> 数据量 < 10K？用 QLoRA。有偏好数据？加 DPO。日请求 > 1000？自部署比 API 便宜。

明天 Day 3 我们要把模型变成产品——Agent、RAG、部署优化。

大家辛苦了，有问题随时找我！"

---

# 附录：教学备忘

## 常见技术问题速查

| 问题 | 解决 |
|------|------|
| PEFT 没装 | `pip install peft` |
| CUDA OOM | 减小 batch_size 或用 CPU |
| 模型下载慢 | 确认 HF_ENDPOINT 设了镜像 |
| tokenizer warning | 可以忽略，代码已处理 |
| DPO 训练 loss 不降 | 学习率太小或数据太少，正常现象 |

## 每个练习的预估完成时间

| 练习 | 难度 | 快的学员 | 慢的学员 | 讲解 |
|------|------|---------|---------|------|
| 练习1: lora_forward | 进阶 | 3分钟 | 8分钟 | 2分钟 |
| 练习2: rank对比 | 入门 | 3分钟 | 6分钟 | 3分钟 |
| 练习3: 显存估算 | 入门 | 3分钟 | 7分钟 | 3分钟 |
| 练习4: DPO对比 | 进阶 | 5分钟 | 10分钟 | 3分钟 |
| 练习5: 评测框架 | 入门 | 3分钟 | 6分钟 | 2分钟 |
| 练习6: 企业决策 | 入门 | 3分钟 | 7分钟 | 5分钟 |

## 时间调控建议

- 如果前面讲慢了，Part 5 评测可以压缩（PPL 快速过，BLEU/F1 跳过）
- 如果前面讲快了，Part 6 企业决策可以多讨论（让学员分享自己公司的场景）
- 练习做不完没关系，给答案让学员课后看
- DPO 公式是难点，宁可多花 5 分钟讲清楚，也别赶进度